# Previsão de Pontuação no Fantasy Premier League

**SME0828 — Introdução à Ciência de Dados** | Prof. Francisco Rodrigues

| Integrante | NUSP |
|:---|:---|
| Artur Kenzo | xxxxxxx |
| Daniel Jorge | 15446861 |
| Luis Henrique | xxxxxxx |

---

**Base de dados:** [`vaastav/Fantasy-Premier-League`](https://github.com/vaastav/Fantasy-Premier-League) - arquivo `merged_gw.csv`

## Questão Fundamental e Hipótese

> "É possível prever com precisão a pontuação de um jogador na próxima rodada da Premier League utilizando seu histórico recente de desempenho e a força defensiva/ofensiva do adversário?"

**Hipótese:** Modelos baseados em árvores (Random Forest e XGBoost/LightGBM) usando médias móveis (rolling features) e métricas de dificuldade do adversário superam modelos lineares e baselines simples, embora a alta variância de eventos raros (gols, cartões) limite o $R^2$.

**Divisão temporal (sem vazamento):**
- **Treino:** Rodadas 1 a 28
- **Teste:** Rodadas 29 a 38

---

## Etapa 1 - Preparação, Limpeza e Engenharia de Atributos (Anti-Leakage)

Nesta etapa realizamos:

1. **[1.1]** Carregamento e ordenação temporal dos dados
2. **[1.2]** Remoção de atributos irrelevantes e tratamento de vazamento (data leakage)
3. **[1.3]** Dados faltantes e rodadas duplas (Double Gameweeks)
4. **[1.4]** Criação de features históricas com médias móveis (janelas de 3 e 5 rodadas)
5. **[1.5]** Features de contexto do adversário (força ofensiva/defensiva histórica)
6. **[1.6]** Validação anti-leakage e resumo final

In [38]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


In [39]:
# Configuração ──────────────────────────────────────────────────────────────
SEASON = '2024-25'

MERGED_GW_URL = (
    'https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/'
    f'master/data/{SEASON}/gws/merged_gw.csv'
)
TEAMS_URL = (
    'https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/'
    f'master/data/{SEASON}/teams.csv'
)

# Janelas para as médias móveis históricas
ROLLING_WINDOWS = [3, 5]

# Colunas de estatísticas de jogo usadas como base para as rolling features
ROLLING_BASE_COLS = [
    'total_points', 'minutes', 'ict_index', 'threat', 'creativity',
    'expected_goals', 'expected_assists', 'goals_scored', 'assists',
    'clean_sheets', 'bonus',
]

print(f'Temporada : {SEASON}')
print(f'URL dados : {MERGED_GW_URL}')

Temporada : 2024-25
URL dados : https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2024-25/gws/merged_gw.csv


### 1.1 - Carregamento e Inspeção Inicial

In [40]:
print(f'Carregando dados da temporada {SEASON}...\n')

df_raw = pd.read_csv(MERGED_GW_URL)
df_teams = pd.read_csv(TEAMS_URL)

n_rows, n_cols = df_raw.shape
n_teams = df_teams.shape[0]
print(f'  merged_gw : {n_rows:,} linhas x {n_cols} colunas')
print(f'  teams     : {n_teams} times')
print('\nColunas disponíveis:')
print(df_raw.columns.tolist())

Carregando dados da temporada 2024-25...

  merged_gw : 27,605 linhas x 49 colunas
  teams     : 20 times

Colunas disponíveis:
['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW']


In [41]:
print('=== Tipos de dados ===')
print(df_raw.dtypes.to_string())
print('\n=== Amostra dos dados ===')
display(df_raw.head(5))
print('\n=== Distribuição do alvo: total_points ===')
print(df_raw['total_points'].describe())
print('\n=== Valores ausentes (colunas com NaN) ===')
missing = df_raw.isnull().sum()
missing_nz = missing[missing > 0].sort_values(ascending=False)
if len(missing_nz) > 0:
    print(missing_nz.to_string())
else:
    print('Nenhum valor ausente encontrado.')

=== Tipos de dados ===
name                              str
position                          str
team                              str
xP                            float64
assists                         int64
bonus                           int64
bps                             int64
clean_sheets                    int64
creativity                    float64
element                         int64
expected_assists              float64
expected_goal_involvements    float64
expected_goals                float64
expected_goals_conceded       float64
fixture                         int64
goals_conceded                  int64
goals_scored                    int64
ict_index                     float64
influence                     float64
kickoff_time                      str
minutes                         int64
mng_clean_sheets              float64
mng_draw                      float64
mng_goals_scored              float64
mng_loss                      float64
mng_underdog_draw          

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,1,0.0,2,0,0,0,50,False,0,1
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,1,0.0,0,0,0,0,45,True,0,1
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,2,0.0,0,0,0,0,50,True,0,1
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,2,0.0,0,0,0,0,45,False,0,1
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,1,16.0,1,0,0,0,55,False,1,1



=== Distribuição do alvo: total_points ===
count    27605.000000
mean         1.200906
std          2.416882
min         -5.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         25.000000
Name: total_points, dtype: float64

=== Valores ausentes (colunas com NaN) ===
mng_clean_sheets     14178
mng_draw             14178
mng_goals_scored     14178
mng_loss             14178
mng_underdog_draw    14178
mng_underdog_win     14178
mng_win              14178


### 1.1 - Ordenação Temporal

É necessário ordenar estritamente por jogador (`element`) e rodada (`round`) para as features históricas que serão criadas a seguir. Sem essa ordenação, as médias móveis com `.shift(1)` capturariam dados de rodadas erradas.

In [42]:
df = df_raw.copy()
df = df.sort_values(['element', 'round']).reset_index(drop=True)

rmin = df['round'].min()
rmax = df['round'].max()
n_jogadores = df['element'].nunique()

print('DataFrame ordenado por [element, round].')
print(f'  Rodadas disponíveis  : {rmin} a {rmax}')
print(f'  Jogadores únicos     : {n_jogadores:,}')
print(f'  Total de observações : {len(df):,}')

DataFrame ordenado por [element, round].
  Rodadas disponíveis  : 1 a 38
  Jogadores únicos     : 804
  Total de observações : 27,605


### 1.2 - Remoção de Atributos e Tratamento de Vazamento (Data Leakage)

Vazamento de dados ocorre quando informações do momento `t` (que ainda não existiriam ao se fazer a previsão) são usadas como features para prever `t`. Os principais casos aqui são:

| Coluna | Problema | Solução |
|:---|:---|:---|
| `team_h_score`, `team_a_score` | Placar do jogo atual | Extrair info útil e remover |
| `xP` | Viés de lookahead documentado no repositório da base de dados | Aplicar `.shift(1)` => `xP_lag1` |
| `name`, `kickoff_time`, `fixture` | Identificadores textuais sem valor preditivo | Remover |

In [43]:
# Derivar gols marcados/sofridos pelo time do jogador em cada partida antes de descartar as colunas de placar
if 'team_h_score' in df.columns and 'team_a_score' in df.columns:
    df['team_goals_scored'] = np.where(
        df['was_home'] == True, df['team_h_score'], df['team_a_score']
    )
    df['team_goals_conceded_match'] = np.where(
        df['was_home'] == True, df['team_a_score'], df['team_h_score']
    )
    print('Derivadas: team_goals_scored | team_goals_conceded_match')

# xP com shift(1): transforma xP[t] em xP da rodada anterior (sem vazamento)
if 'xP' in df.columns:
    df['xP_lag1'] = df.groupby('element')['xP'].shift(1)
    print('Derivada : xP_lag1 (xP defasada 1 rodada)')

# Extrair estatísticas por time e partida (fixture) antes de descartar 'fixture'
# Preserva informações de Double Gameweeks (uma linha por jogo, não por rodada)
if 'fixture' in df.columns:
    team_game_stats = (
        df[['team', 'round', 'fixture', 'team_goals_scored', 'team_goals_conceded_match']]
        .drop_duplicates(subset=['team', 'fixture'])
        .sort_values(['team', 'round'])
        .reset_index(drop=True)
    )
    print(f'\nEstatísticas de jogo extraídas: {len(team_game_stats)} registros (time x partida)')
else:
    team_game_stats = (
        df[['team', 'round', 'team_goals_scored', 'team_goals_conceded_match']]
        .drop_duplicates(subset=['team', 'round'])
        .sort_values(['team', 'round'])
        .reset_index(drop=True)
    )
    print(f'\nEstatísticas de rodada extraídas: {len(team_game_stats)} registros')

# Remover colunas com vazamento direto ou unicamente identificadoras
COLS_TO_DROP = [
    'name',                        # identificador textual (mantemos element)
    'kickoff_time',                # após ordenação, não agrega informação
    'fixture',                     # redundante com (round, opponent_team)
    'team_h_score',                # placar do jogo atual => vazamento direto
    'team_a_score',                # placar do jogo atual => vazamento direto
    'xP',                          # substituído por xP_lag1
    'team_goals_scored',           # resultado do jogo atual => vazamento
    'team_goals_conceded_match',   # resultado do jogo atual => vazamento
]
cols_removed = [c for c in COLS_TO_DROP if c in df.columns]
df.drop(columns=cols_removed, inplace=True)

print(f'\nColunas removidas: {cols_removed}')
print(f'Shape atual: {df.shape}')

Derivadas: team_goals_scored | team_goals_conceded_match
Derivada : xP_lag1 (xP defasada 1 rodada)

Estatísticas de jogo extraídas: 760 registros (time x partida)

Colunas removidas: ['name', 'kickoff_time', 'fixture', 'team_h_score', 'team_a_score', 'xP', 'team_goals_scored', 'team_goals_conceded_match']
Shape atual: (27605, 44)


### 1.3 - Dados Faltantes e Rodadas Duplas (*Double Gameweeks*)

Dois casos especiais precisam de atenção:

- **Jogadores no banco (`minutes == 0`):** estatísticas de jogo ausentes são zero, então colocamos `0`.
- **Double Gameweeks:** jogadores jogam duas partidas na mesma rodada, gerando duas linhas para o mesmo `(element, round)`. Resolvemos somando as estatísticas acumulativas (pontos, minutos, gols…) e mantendo o último valor para atributos contextuais (posição, preço, adversário).

In [44]:
# Preenchimento de zeros para jogadores que não jogaram
STAT_COLS_ZERO = [
    'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals',
    'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves',
    'bonus', 'bps', 'expected_goals', 'expected_assists',
    'expected_goal_involvements', 'expected_goals_conceded',
    'influence', 'creativity', 'threat', 'ict_index',
]
stat_zero_present = [c for c in STAT_COLS_ZERO if c in df.columns]
mask_bench = df['minutes'] == 0
n_imputed = 0
for col in stat_zero_present:
    n_nan = df.loc[mask_bench, col].isna().sum()
    if n_nan > 0:
        df.loc[mask_bench & df[col].isna(), col] = 0
        n_imputed += n_nan
print(f'Imputação com 0 (jogadores no banco): {n_imputed} valores preenchidos')

# Rodadas Duplas
double_gw = df.groupby(['element', 'round']).size()
double_gw_cases = double_gw[double_gw > 1]
n_double = len(double_gw_cases)
print(f'\nDouble Gameweeks detectados: {n_double} pares (element, round)')
if n_double > 0:
    print('\nAmostra:')
    print(double_gw_cases.head(5).to_string())

# Agregação: somar estatísticas acumulativas; 'last' para atributos contextuais
SUM_COLS = {
    'total_points', 'minutes', 'goals_scored', 'assists', 'clean_sheets',
    'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
    'red_cards', 'saves', 'bonus', 'bps', 'goals_conceded',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'expected_goals_conceded', 'transfers_in', 'transfers_out', 'transfers_balance',
}
agg_dict = {}
for col in df.columns:
    if col in ('element', 'round'):
        continue
    agg_dict[col] = 'sum' if col in SUM_COLS else 'last'

df = df.groupby(['element', 'round'], as_index=False).agg(agg_dict)
df = df.sort_values(['element', 'round']).reset_index(drop=True)

n_nan_total = df.isnull().sum().sum()
print(f'\nShape após tratamento: {df.shape}')
print(f'NaN totais restantes : {n_nan_total}')

Imputação com 0 (jogadores no banco): 0 valores preenchidos

Double Gameweeks detectados: 374 pares (element, round)

Amostra:
element  round
1        33       2
2        33       2
3        33       2
4        33       2
5        33       2

Shape após tratamento: (27231, 44)
NaN totais restantes : 100048


### 1.4 - Features Históricas com Médias Móveis (Rolling Features)

Para cada estatística de jogo $f$ e janela $w \in \{3, 5\}$ rodadas, criamos:

$$f_{\text{roll}w}[t] = \frac{1}{w} \sum_{k=1}^{w} f[t-k]$$

O `.shift(1)` antes do `.rolling()` garante que a janela use apenas rodadas anteriores a $t$, eliminando qualquer vazamento da rodada atual. `min_periods=1` permite calcular mesmo nas primeiras rodadas (histórico insuficiente).

In [45]:
rolling_base_present = [c for c in ROLLING_BASE_COLS if c in df.columns]

for feat in rolling_base_present:
    for w in ROLLING_WINDOWS:
        col_name = f'{feat}_roll{w}'
        df[col_name] = (
            df.groupby('element')[feat]
            .transform(lambda x, window=w: x.shift(1).rolling(window, min_periods=1).mean())
        )

# Lag-1 direto: pontuação e minutos da rodada imedatamente anterior
for feat in ['total_points', 'minutes']:
    if feat in df.columns:
        df[f'{feat}_lag1'] = df.groupby('element')[feat].shift(1)

created_roll = [f'{f}_roll{w}' for f in rolling_base_present for w in ROLLING_WINDOWS]
created_lag  = [f'{f}_lag1' for f in ['total_points', 'minutes'] if f in df.columns]

print(f'Rolling features criadas : {len(created_roll)}')
print(f'  Exemplo: {created_roll[:4]}')
print(f'Lag features criadas     : {len(created_lag)}')
print(f'  Exemplo: {created_lag}')
print(f'\nShape atual: {df.shape}')

Rolling features criadas : 22
  Exemplo: ['total_points_roll3', 'total_points_roll5', 'minutes_roll3', 'minutes_roll5']
Lag features criadas     : 2
  Exemplo: ['total_points_lag1', 'minutes_lag1']

Shape atual: (27231, 68)


### 1.5 - Features de Contexto do Adversário

Para cada adversário e rodada $t$, calculamos a média histórica acumulada (até $t-1$) de:

- `opp_avg_goals_scored` => força ofensiva do adversário (gols marcados por jogo)
- `opp_avg_goals_conceded` => vulnerabilidade defensiva do adversário (gols sofridos por jogo)

Usamos `expanding().mean()` com `.shift(1)` sobre a série de cada time, de forma que apenas informação histórica entra na feature.

In [46]:
team_game_stats = team_game_stats.sort_values(['team', 'round'])

team_game_stats['opp_avg_goals_scored'] = (
    team_game_stats.groupby('team')['team_goals_scored']
    .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
)
team_game_stats['opp_avg_goals_conceded'] = (
    team_game_stats.groupby('team')['team_goals_conceded_match']
    .transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
)

# Converter o nome do time (texto) para o ID usado em opponent_team
team_id_map = df_teams.set_index('name')['id'].to_dict()
team_game_stats['team_id'] = team_game_stats['team'].map(team_id_map)

if team_game_stats['team_id'].isna().any():
    nomes_sem_id = sorted(team_game_stats.loc[team_game_stats['team_id'].isna(), 'team'].dropna().unique())
    print(f'Aviso: times sem ID encontrado em df_teams: {nomes_sem_id}')

# Agregar por (team_id, round) para lidar com Double Gameweeks na série de times
opp_context = (
    team_game_stats
    .dropna(subset=['team_id'])
    .groupby(['team_id', 'round'])[['opp_avg_goals_scored', 'opp_avg_goals_conceded']]
    .mean()
    .reset_index()
    .rename(columns={'team_id': 'opponent_team'})
)

# Garantir que ambas as chaves tenham o mesmo tipo inteiro e a mesma codificação de ID usada pela base FPL
df['opponent_team'] = pd.to_numeric(df['opponent_team'], errors='coerce').astype('Int64')
opp_context['opponent_team'] = pd.to_numeric(opp_context['opponent_team'], errors='coerce').astype('Int64')
df['round'] = pd.to_numeric(df['round'], errors='coerce').astype('int64')
opp_context['round'] = pd.to_numeric(opp_context['round'], errors='coerce').astype('int64')

# Merge: o opponent_team de um jogador na rodada t recebe o contexto histórico do adversário
df = df.merge(opp_context, on=['opponent_team', 'round'], how='left', validate='many_to_one')

print('Features de contexto do adversário adicionadas:')
print('  opp_avg_goals_scored   -> força ofensiva do adversário')
print('  opp_avg_goals_conceded -> vulnerabilidade defensiva do adversário')
print(f'\nShape final: {df.shape}')
print(f"Linhas sem contexto do adversário: {df['opp_avg_goals_scored'].isna().sum():,}")
display(
    df[['element', 'round', 'opponent_team', 'was_home',
        'opp_avg_goals_scored', 'opp_avg_goals_conceded']].head(10)
)

Features de contexto do adversário adicionadas:
  opp_avg_goals_scored   -> força ofensiva do adversário
  opp_avg_goals_conceded -> vulnerabilidade defensiva do adversário

Shape final: (27231, 70)
Linhas sem contexto do adversário: 616


,element,round,opponent_team,was_home,opp_avg_goals_scored,opp_avg_goals_conceded
0,1,1,20,True,NaN,NaN
1,1,2,2,False,2.000000,1.000000
2,1,3,5,True,2.500000,0.500000
3,1,4,18,False,2.000000,1.000000
4,1,5,13,False,2.750000,0.750000
5,1,6,11,True,1.200000,1.600000
6,1,7,17,True,0.500000,2.000000
7,1,8,3,False,1.142857,1.428571
8,1,9,12,True,1.875000,0.375000
9,1,10,15,False,1.000000,1.111111


### 1.6 - Validação Anti-Leakage

Verificamos automaticamente que `total_points_lag1[i]` é exatamente `total_points[i-1]` para cada jogador, confirmando que nenhuma informação da rodada atual contaminou as features históricas.

In [47]:
# Exibir a série de um jogador específico para inspeção visual
top_player = df['element'].value_counts().idxmax()
player_df = (
    df[df['element'] == top_player]
    [['element', 'round', 'total_points', 'total_points_lag1',
      'total_points_roll3', 'total_points_roll5']]
    .head(10)
    .reset_index(drop=True)
)
print(f'Verificação anti-leakage — jogador element={top_player}:')
print('Esperado: total_points_lag1[i] == total_points[i-1]\n')
display(player_df)

# Checagem automática para todos os jogadores
df_sorted = df.sort_values(['element', 'round'])

def check_lag(g):
    if len(g) < 2:
        return True
    lag  = g['total_points_lag1'].iloc[1:].values
    prev = g['total_points'].iloc[:-1].values
    mask = ~(pd.isna(lag) & pd.isna(prev))
    if mask.sum() == 0:
        return True
    return np.allclose(lag[mask], prev[mask])

lag_ok = df_sorted.groupby('element').apply(check_lag)
all_ok = lag_ok.all()
status = 'OK - sem vazamento detectado' if all_ok else 'ERRO - verificar pipeline!'
print(f'\nConsistência do lag1 para todos os jogadores: {status}')

Verificação anti-leakage — jogador element=10:
Esperado: total_points_lag1[i] == total_points[i-1]



,element,round,total_points,total_points_lag1,total_points_roll3,total_points_roll5
0,10,1,0,NaN,NaN,NaN
1,10,2,1,0.0,0.000000,0.000000
2,10,3,0,1.0,0.500000,0.500000
3,10,4,0,0.0,0.333333,0.333333
4,10,5,7,0.0,0.333333,0.250000
5,10,6,1,7.0,2.333333,1.600000
6,10,7,4,1.0,2.666667,1.800000
7,10,8,1,4.0,4.000000,2.400000
8,10,9,1,1.0,2.000000,2.600000
9,10,10,2,1.0,2.000000,2.800000



Consistência do lag1 para todos os jogadores: OK - sem vazamento detectado


### 1.7 - Resumo Final da Etapa 1

In [48]:
rolling_cols  = [c for c in df.columns if '_roll' in c or '_lag1' in c]
context_cols  = [c for c in ['was_home', 'opp_avg_goals_scored', 'opp_avg_goals_conceded'] if c in df.columns]
pre_game_cols = [c for c in ['value', 'selected', 'transfers_in', 'transfers_out',
                              'transfers_balance', 'xP_lag1'] if c in df.columns]

rmin = df['round'].min()
rmax = df['round'].max()
n_jog = df['element'].nunique()

print('=' * 56)
print('  RESUMO - PASSO 1: PREPARAÇÃO DOS DADOS')
print('=' * 56)
print(f'\nShape final              : {df.shape}')
print(f'Rodadas                  : {rmin} a {rmax}')
print(f'Jogadores únicos         : {n_jog:,}')
print(f'\nColunas identificadoras  : element, round, team, position, opponent_team')
print(f'Target                   : total_points')
print(f'Features pré-jogo ({len(pre_game_cols)})   : {pre_game_cols}')
print(f'Rolling / lag ({len(rolling_cols)})       : {rolling_cols[:6]}...')
print(f'Contexto adversário ({len(context_cols)}) : {context_cols}')
print(f'\nNaN em rolling features (1ª rodada sem histórico suficiente):')
nan_rolling = df[rolling_cols].isnull().sum()
nan_nz = nan_rolling[nan_rolling > 0]
if len(nan_nz) > 0:
    print(nan_nz.to_string())
else:
    print('  Nenhum NaN detectado.')

# Salvar o dataframe preparado em CSV para uso nos próximos passos
df.to_csv('fpl_prepared.csv', index=False)
print(f'\nDados salvos em fpl_prepared.csv')

  RESUMO - PASSO 1: PREPARAÇÃO DOS DADOS

Shape final              : (27231, 70)
Rodadas                  : 1 a 38
Jogadores únicos         : 804

Colunas identificadoras  : element, round, team, position, opponent_team
Target                   : total_points
Features pré-jogo (6)   : ['value', 'selected', 'transfers_in', 'transfers_out', 'transfers_balance', 'xP_lag1']
Rolling / lag (25)       : ['xP_lag1', 'total_points_roll3', 'total_points_roll5', 'minutes_roll3', 'minutes_roll5', 'ict_index_roll3']...
Contexto adversário (3) : ['was_home', 'opp_avg_goals_scored', 'opp_avg_goals_conceded']

NaN em rolling features (1ª rodada sem histórico suficiente):
xP_lag1                   802
total_points_roll3        804
total_points_roll5        804
minutes_roll3             804
minutes_roll5             804
ict_index_roll3           804
ict_index_roll5           804
threat_roll3              804
threat_roll5              804
creativity_roll3          804
creativity_roll5          804
expect